# Use API for analysis rather than downloading as storage
- https://www.tycho.pitt.edu/dataset/api/

In [23]:
import os
from dotenv import load_dotenv

# Load variables from .env file into the environment
load_dotenv()

# Access the variables
API_KEY = os.getenv("TYCHO_API_KEY")

### Columns needed from this query:
- ConditionName
- ConditionSNOMED
- CountryCode
- Admin1ISO
- Admin1Name
- PeriodStartDate
- PeriodEndDate
- CountValue
- PartOfCumulativeCountSeries
- Fatalities

In [24]:
core_columns = [
    "ConditionName",
    "CountryCode",
    "Admin1ISO",
    "PeriodEndDate",
    "CountValue",
    "PartOfCumulativeCountSeries",
    "Fatalities"
]

In [25]:
import pandas as pd

disease = "Pertussis"

url = (
    f"https://www.tycho.pitt.edu/api/query?"
    f"apikey={API_KEY}"
    f"&ConditionName={disease}"
    f"&CountryISO=US"
    f"&Admin1ISO=US-PA"
    f"&PeriodStartDate%3E=1960-01-01"
    f"&PeriodEndDate%3C=2026-12-31"
)

df = pd.read_csv(url)[core_columns]

print(df.shape)
df.head()

(2921, 7)


,ConditionName,CountryCode,Admin1ISO,PeriodEndDate,CountValue,PartOfCumulativeCountSeries,Fatalities
0,Pertussis,US,US-PA,1975-05-03,3,0,0
1,Pertussis,US,US-PA,1975-05-10,1,0,0
2,Pertussis,US,US-PA,1975-06-21,1,0,0
3,Pertussis,US,US-PA,1975-07-05,1,0,0
4,Pertussis,US,US-PA,1975-07-26,1,0,0


In [ ]:
df['PeriodEndDate'].max()

'1975-05-03'

In [27]:
df.to_csv(f"../raw/tycho/tycho_{disease}.csv", index=False)

In [28]:
assert False

AssertionError: 

In [ ]:
import pandas as pd

tp_1 = pd.read_table('../misc/genecounts1.txt')
tp_2 = pd.read_table('../misc/genecounts2.txt')

tps_df = pd.merge(tp_1, tp_2, on=' gene')


tps_df.rename(columns={'0_x': 'tp_1', '0_y': 'tp_2', ' gene': 'gene'}, inplace=True)
tps_df = tps_df.dropna()
tps_df

,tp_1,gene,tp_2
6,1079,dnaA,1079
7,35,dnaN,4
8,35,dnaN,0
9,0,yaaA,0
10,3983,recF,0
...,...,...,...
2217,0,yidC,0
2218,0,yidC,1
2219,0,rnpA,0
2220,0,rpmH,0


In [ ]:
tps_df['diff'] = abs(tps_df['tp_1'] - tps_df['tp_2'])
tps_df

,tp_1,gene,tp_2,diff
10,3983,recF,0,3983
6,1079,dnaA,1079,0
1911,880,ackA,588,292
234,524,cysS,134,390
1603,372,ftsA,488,116
...,...,...,...,...
1106,0,flgB,0,0
1105,0,flgK,0,0
1104,0,cheR,0,0
1103,0,fliY,0,0


In [ ]:
tps_df = tps_df[(tps_df['tp_1'] > 0) & (tps_df['tp_2'] > 0)]

In [ ]:
tps_df = tps_df.sort_values(by='diff', ascending=False)
tps_df.head(10)

,tp_1,gene,tp_2,diff
12,52,gyrA,7832,7780
234,524,cysS,134,390
1911,880,ackA,588,292
940,30,prpE,249,219
11,215,gyrB,22,193
1831,80,folC,262,182
229,201,disA,51,150
1261,23,spoVS,153,130
228,166,radA,42,124
753,125,cadA,3,122


In [ ]:
import plotly.express as px

plot_df = tps_df.head(10).melt(
    id_vars="gene",
    value_vars=["tp_1", "tp_2"],
    var_name="time_point",
    value_name="counts"
)

fig = px.bar(
    plot_df,
    x="gene",
    y="counts",
    color="time_point",
    barmode="group",
    text="counts",
    title="Top 10 Genes by Count",
    labels={
        "gene": "Gene",
        "counts": "Count",
        "time_point": "Time Point",
    },
)

fig.update_traces(textposition="outside")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Gene",
    yaxis_title="Count",
    legend_title="Time Point",
)

fig.show()